# N-gram Smoothing Strategies: Analysis Notebook

This notebook loads the experiment results and provides interactive exploration of the smoothing comparison data.

In [ ]:
import csv
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = Path("../results")
PLOTS_DIR = Path("../plots")

In [ ]:
# Load results
rows = []
with open(RESULTS_DIR / "results.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        row["fraction"] = float(row["fraction"])
        row["n_train_tokens"] = int(row["n_train_tokens"])
        row["vocab_size"] = int(row["vocab_size"])
        row["order"] = int(row["order"])
        row["perplexity"] = float(row["perplexity"]) if row["perplexity"] != "inf" else float("inf")
        row["zero_prob_rate"] = float(row["zero_prob_rate"])
        row["oov_rate"] = float(row["oov_rate"])
        row["perplexity_rare"] = float(row["perplexity_rare"]) if row["perplexity_rare"] != "inf" else float("inf")
        rows.append(row)

print(f"Loaded {len(rows)} result rows")

In [ ]:
# Summary table: best method per corpus size
fracs = sorted(set(r["fraction"] for r in rows))
for order in [2, 3]:
    print(f"\n{'Bigram' if order == 2 else 'Trigram'} - Best method per corpus fraction:")
    print(f"{'Fraction':>10} {'Best Method':>22} {'Perplexity':>12}")
    print("-" * 48)
    for frac in fracs:
        subset = [r for r in rows if r["order"] == order and r["fraction"] == frac]
        valid = [r for r in subset if not math.isinf(r["perplexity"])]
        if valid:
            best = min(valid, key=lambda r: r["perplexity"])
            print(f"{frac:>10.2f} {best['method']:>22} {best['perplexity']:>12.1f}")
        else:
            print(f"{frac:>10.2f} {'(all inf)':>22}")

In [ ]:
# Good-Turing stability analysis
gt_rows = [r for r in rows if r["method"] == "GoodTuring"]
print("Good-Turing stability flag:")
for r in sorted(gt_rows, key=lambda r: r["n_train_tokens"]):
    print(f"  frac={r['fraction']:.2f}  order={r['order']}  unstable={r['gt_unstable']}  ppl={r['perplexity']:.1f}")

In [ ]:
# Kneser-Ney discount evolution
kn_rows = [r for r in rows if r["method"] == "KneserNey" and r["order"] == 2]
kn_rows.sort(key=lambda r: r["n_train_tokens"])
print("Kneser-Ney discount parameters (bigram):")
for r in kn_rows:
    print(f"  frac={r['fraction']:.2f}  d1={r['kn_d1']}  d2={r['kn_d2']}  d3={r['kn_d3']}")

In [ ]:
# Display saved plots
plot_files = sorted(PLOTS_DIR.glob("*.png"))
for pf in plot_files:
    print(f"\n--- {pf.name} ---")
    img = plt.imread(pf)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img)
    ax.axis("off")
    plt.show()